A1 Cleaning Log


A3 -join map


A4- audit

A6-Feature engineering table


1 imports and paths

In [1]:
import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="colorblind")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
raw          = PROJECT_ROOT / "data" / "raw"
processed    = PROJECT_ROOT / "data" / "processed"
figures_dir  = PROJECT_ROOT / "figures"
reports      = PROJECT_ROOT / "reports"
submission_dir = PROJECT_ROOT / "submission"
models_dir   = PROJECT_ROOT / "models"
for p in [processed, figures_dir, reports, submission_dir, models_dir]:
    p.mkdir(parents=True, exist_ok=True)

TEAM_NAME = "dawit"
print("Project root:", PROJECT_ROOT)

Project root: c:\Users\sclab\Desktop\A hackaton\notebooks


In [2]:
from pathlib import Path
import pandas as pd

# Anchor to project root regardless of where the kernel starts
ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")
RAW  = ROOT / "data" / "raw"

print("RAW exists:", RAW.exists())
print("CSVs:", [f.name for f in RAW.glob("*.csv")])

RAW exists: True
CSVs: ['crop_yield_leaderboard_test.csv', 'crop_yield_train.csv', 'market_prices.csv', 'regional_weather.csv', 'submission_template.csv']


In [3]:
from pathlib import Path

names = {
    "crop_yield_train.csv",
    "crop_yield_leaderboard_test.csv",
    "submission_template.csv",
    "regional_weather.csv",
    "prices.csv",              # the file from before, just in case
}

root = Path.cwd()
# walk up to the "A hackaton" folder (or Desktop if not found)
base = root
while base.name.lower() != "a hackaton" and base.parent != base:
    base = base.parent

print("Project root:", base)
print("\nFiles found:")
for p in base.rglob("*"):
    if p.is_file() and p.name in names:
        print("  ", p)

Project root: c:\Users\sclab\Desktop\A hackaton

Files found:
   c:\Users\sclab\Desktop\A hackaton\data\raw\crop_yield_leaderboard_test.csv
   c:\Users\sclab\Desktop\A hackaton\data\raw\crop_yield_train.csv
   c:\Users\sclab\Desktop\A hackaton\data\raw\regional_weather.csv
   c:\Users\sclab\Desktop\A hackaton\data\raw\submission_template.csv


2 load raw files

In [4]:
from pathlib import Path
import pandas as pd

raw = Path(r"C:\Users\sclab\Desktop\A hackaton\data\raw")   # <-- the real folder

print("raw =", raw)
print("exists:", raw.exists())
print("files:", [f.name for f in raw.glob("*.csv")])

train    = pd.read_csv(raw / "crop_yield_train.csv")
test     = pd.read_csv(raw / "crop_yield_leaderboard_test.csv")
template = pd.read_csv(raw / "submission_template.csv")
weather  = pd.read_csv(raw / "regional_weather.csv")

raw = C:\Users\sclab\Desktop\A hackaton\data\raw
exists: True
files: ['crop_yield_leaderboard_test.csv', 'crop_yield_train.csv', 'market_prices.csv', 'regional_weather.csv', 'submission_template.csv']


In [5]:
prices = pd.read_csv(raw / "market_prices.csv")

print("prices shape:", prices.shape)
print("prices columns:", prices.columns.tolist())
print(prices.head())

prices shape: (100, 4)
prices columns: ['crop_type', 'region', 'year', 'price_birr_per_quintal']
   crop_type  region  year  price_birr_per_quintal
0   sorghum   Amhara  2022                     NaN
1      wheat   SNNPR  2023                  5321.0
2      maize  Amhara  2022                  2824.0
3      Wheat  Tigray  2022                  4686.0
4    sorghum  Somali  2023                  3642.0


In [6]:
from pathlib import Path
ROOT = Path(r"C:\Users\sclab\Desktop\A hackaton")
for p in ROOT.rglob("*.csv"):
    print(p.relative_to(ROOT))

reports\A1_cleaning_log.csv
reports\A2_label_proof.csv
reports\A5_join_proof.csv
reports\A6_feature_table.csv
submission\team_dawit_submission.csv
notebooks\reports\A1_cleaning_log.csv
notebooks\reports\A2_label_proof.csv
notebooks\reports\A5_join_proof.csv
notebooks\reports\A6_feature_table.csv
notebooks\reports\feature_importance.csv
notebooks\reports\validation_preds.csv
notebooks\submission\team_dawit_submission.csv
notebooks\data\processed\data_dictionary_master.csv
notebooks\data\processed\master_test.csv
notebooks\data\processed\master_train.csv
data\processed\data_dictionary_master.csv
data\processed\master_test.csv
data\processed\master_train.csv
data\raw\crop_yield_leaderboard_test.csv
data\raw\crop_yield_train.csv
data\raw\market_prices.csv
data\raw\regional_weather.csv
data\raw\submission_template.csv
.venv\Lib\site-packages\tornado\test\csv_translations\fr_FR.csv
.venv\Lib\site-packages\sklearn\datasets\data\breast_cancer.csv
.venv\Lib\site-packages\sklearn\datasets\data\i

Cell 3 — snapshot "before" state for A2 proof

In [7]:
before_regions_train    = sorted(train["region"].dropna().astype(str).unique())
before_crops_train      = sorted(train["crop_type"].dropna().astype(str).unique())
before_regions_weather  = sorted(weather["region"].dropna().astype(str).unique())
before_regions_prices   = sorted(prices["region"].dropna().astype(str).unique())
before_crops_prices     = sorted(prices["crop_type"].dropna().astype(str).unique())
print("BEFORE train regions:", before_regions_train)
print("BEFORE train crops:", before_crops_train)
print("BEFORE weather regions:", before_regions_weather)

BEFORE train regions: ['AMHARA', 'Amhara', 'Amhara ', 'OROMIA', 'Oromia', 'Oromia ', 'SNNPR', 'SNNPR ', 'SOMALI', 'Somali', 'Somali ', 'TIGRAY', 'Tigray', 'Tigray ', 'amhara', 'oromia', 'snnpr', 'somali', 'tigray']
BEFORE train crops: [' barley ', ' maize ', ' sorghum ', ' teff ', ' wheat ', 'BARLEY', 'Barley', 'MAIZE', 'Maize', 'SORGHUM', 'Sorghum', 'TEFF', 'Tef', 'Teff', 'WHEAT', 'Wheat', 'barley', 'maize', 'sorghum', 'teff', 'wheat']
BEFORE weather regions: ['AMH', 'Amhara', 'Amhara ', 'ORO', 'Oromia', 'Oromia ', 'SNNP', 'SNNPR', 'SNNPR ', 'SOM', 'Somali', 'Somali ', 'TIG', 'Tigray', 'Tigray ', 'amhara', 'oromia', 'snnpr', 'somali', 'tigray']


In [8]:
raw = Path(r"C:\Users\sclab\Desktop\A hackaton\data\raw")
exists: True
files: ['crop_yield_leaderboard_test.csv', 'crop_yield_train.csv', 'regional_weather.csv', 'submission_template.csv']

Cell 4 — clean labels

In [9]:
def clean_region(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().lower()
    if x.startswith("orom") or x == "oro": return "Oromia"
    if x.startswith("amha") or x == "amh": return "Amhara"
    if x.startswith("snn"): return "SNNPR"
    if x.startswith("tigr") or x == "tig": return "Tigray"
    if x.startswith("soma") or x == "som": return "Somali"
    return x.title()

def clean_crop(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().lower()
    if x in ["teff", "tef"]: return "teff"
    if "wheat" in x: return "wheat"
    if "maize" in x: return "maize"
    if "sorghum" in x: return "sorghum"
    if "barley" in x: return "barley"
    return x

for df in [train, test, weather, prices]:
    if "region" in df.columns:
        df["region"] = df["region"].apply(clean_region)
    if "crop_type" in df.columns:
        df["crop_type"] = df["crop_type"].apply(clean_crop)

print("AFTER train regions:", sorted(train["region"].dropna().unique()))
print("AFTER train crops:",   sorted(train["crop_type"].dropna().unique()))
print("AFTER weather regions:", sorted(weather["region"].dropna().unique()))
print("AFTER prices regions:", sorted(prices["region"].dropna().unique()))
print("AFTER prices crops:",   sorted(prices["crop_type"].dropna().unique()))

AFTER train regions: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']
AFTER train crops: ['barley', 'maize', 'sorghum', 'teff', 'wheat']
AFTER weather regions: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']
AFTER prices regions: ['Amhara', 'Oromia', 'SNNPR', 'Somali', 'Tigray']
AFTER prices crops: ['barley', 'maize', 'sorghum', 'teff', 'wheat']


Cell 5 — replace -999 sentinels

In [10]:
for df in [train, test]:
    df.replace([-999, -999.0, -999.00], np.nan, inplace=True)

print(train.isna().sum())

plot_id                    0
region                     0
crop_type                  0
survey_year                0
planting_month             0
altitude_m                 0
rainfall_mm_season       449
farm_size_ha               0
fertilizer_kg_per_ha     748
improved_seed_used         0
pest_disease_flag        300
soil_quality_index       608
labor_days_per_ha        488
distance_to_market_km      0
yield_tons_per_ha          0
dtype: int64


Cell 6 — outlier capping (fit on train only)

In [11]:
cap_cols = ["farm_size_ha","fertilizer_kg_per_ha","distance_to_market_km",
            "rainfall_mm_season","altitude_m","labor_days_per_ha"]
caps = {c: float(train[c].quantile(0.99)) for c in cap_cols}
print("Caps:", caps)

for c, cap in caps.items():
    train[c] = train[c].clip(upper=cap)
    test[c]  = test[c].clip(upper=cap)

Caps: {'farm_size_ha': 4.262495503276447, 'fertilizer_kg_per_ha': 136.04985626055972, 'distance_to_market_km': 54.93301599047047, 'rainfall_mm_season': 1406.0986457926153, 'altitude_m': 3100.0, 'labor_days_per_ha': 80.20595017515805}


7.planting month number

In [12]:
month_map = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
             "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
for df in [train, test]:
    df["planting_month_num"] = df["planting_month"].astype(str).str.strip().str[:3].str.lower().map(month_map)

8.weather clean + dedupe

In [13]:
weather["month_num"] = weather["month"].astype(str).str.strip().str[:3].str.lower().map(month_map)
for c in ["avg_temp_c","monthly_rainfall_mm","extreme_heat_days"]:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")

weather_before_rows = len(weather)
weather = weather.drop_duplicates(subset=["region","year","month_num"], keep="first")
weather_after_rows = len(weather)
print(f"Weather dedup: {weather_before_rows} → {weather_after_rows}")

Weather dedup: 232 → 226


9.growing-season planting month +3

In [14]:
def make_season_rows(df):
    rows = []
    for _, r in df.iterrows():
        for off in range(4):
            m = r["planting_month_num"] + off
            y = r["survey_year"] + (m - 1) // 12
            mm = (m - 1) % 12 + 1
            rows.append({"plot_id": r["plot_id"], "region": r["region"],
                         "year": y, "month_num": mm})
    return pd.DataFrame(rows)

season_train = make_season_rows(train)
season_test  = make_season_rows(test)

weather_small = weather[["region","year","month_num","avg_temp_c",
                         "monthly_rainfall_mm","extreme_heat_days"]]

season_train = season_train.merge(weather_small, on=["region","year","month_num"], how="left")
season_test  = season_test.merge(weather_small, on=["region","year","month_num"], how="left")

# A4 audit: how many plots had fewer than 4 months matched?
matched_train = season_train.groupby("plot_id")["avg_temp_c"].count()
print("Train plots with < 4 months matched:", int((matched_train < 4).sum()))
matched_test = season_test.groupby("plot_id")["avg_temp_c"].count()
print("Test plots with < 4 months matched:", int((matched_test < 4).sum()))

Train plots with < 4 months matched: 2925
Test plots with < 4 months matched: 748


10.aggregate weather to plot level +join

In [15]:
def agg_weather(season):
    return season.groupby("plot_id").agg(
        season_mean_temp=("avg_temp_c","mean"),
        season_total_rain=("monthly_rainfall_mm","sum"),
        season_extreme_heat=("extreme_heat_days","sum"),
        season_months_count=("avg_temp_c","count"),
    ).reset_index()

agg_train = agg_weather(season_train)
agg_test  = agg_weather(season_test)

train_before_join = len(train)
test_before_join  = len(test)
train = train.merge(agg_train, on="plot_id", how="left")
test  = test.merge(agg_test,  on="plot_id", how="left")
assert len(train) == train_before_join and len(test) == test_before_join
print("Row count unchanged by weather join:", len(train), len(test))
print("Train unmatched after weather join:", int(train["season_mean_temp"].isna().sum()))
print("Test  unmatched after weather join:", int(test["season_mean_temp"].isna().sum()))

Row count unchanged by weather join: 15090 3750
Train unmatched after weather join: 0
Test  unmatched after weather join: 0


11. region-year weather aggregates + derived features

In [16]:
reg_year = weather.groupby(["region","year"]).agg(
    region_year_temp=("avg_temp_c","mean"),
    region_year_rain=("monthly_rainfall_mm","sum"),
).reset_index()

train = train.merge(reg_year, left_on=["region","survey_year"],
                    right_on=["region","year"], how="left").drop(columns=["year"])
test  = test.merge(reg_year, left_on=["region","survey_year"],
                   right_on=["region","year"], how="left").drop(columns=["year"])

for df in [train, test]:
    df["temp_dev"]    = df["season_mean_temp"] - df["region_year_temp"]
    df["rain_dev"]    = df["season_total_rain"] - df["region_year_rain"]
    df["fert_x_seed"] = df["fertilizer_kg_per_ha"] * df["improved_seed_used"]

print(train[["season_mean_temp","season_total_rain","temp_dev","rain_dev","fert_x_seed"]].describe())

       season_mean_temp  season_total_rain      temp_dev      rain_dev  \
count      15090.000000       15090.000000  15090.000000  15090.000000   
mean          20.400557         367.354049     -0.287149   -493.629649   
std            4.436420         195.171450      0.938949    232.012844   
min           15.025000          52.100000     -1.930000  -1055.400000   
25%           17.250000         216.600000     -1.109091   -669.200000   
50%           19.066667         352.100000     -0.438636   -499.400000   
75%           21.350000         477.700000      0.558333   -285.200000   
max           31.175000         924.100000      1.490909    -77.000000   

        fert_x_seed  
count  14342.000000  
mean      15.181519  
std       25.720468  
min        0.000000  
25%        0.000000  
50%        0.000000  
75%       24.246439  
max      136.049856  


12.A1 cleaning log table (markdown)

In [17]:
cleaning_log = pd.DataFrame([
    ["plot", "region", "label inconsistency",
     f"~{len(before_regions_train)} distinct spellings", "100%",
     "Standardized to 5 labels via clean_region()", "Required by A2 and joins"],

    ["plot", "crop_type", "label inconsistency",
     f"~{len(before_crops_train)} distinct spellings", "100%",
     "Standardized to 5 labels via clean_crop()", "Required by A2 and joins"],

    ["plot", "pest_disease_flag", "sentinel -999",
     int((pd.read_csv(raw / 'crop_yield_train.csv')['pest_disease_flag'] == -999).sum()),
     "<1%",
     "Replaced with NaN", "Sentinel is not a valid 0/1 flag"],

    ["plot",
     "rainfall_mm_season, fertilizer_kg_per_ha, soil_quality_index, labor_days_per_ha",
     "missing values", 449 + 748 + 608 + 488, "~15%",
     "Imputed later in pipeline with train median",
     "Cannot drop rows, must predict every plot"],

    ["plot",
     "farm_size_ha, fertilizer_kg_per_ha, distance_to_market_km, rainfall_mm_season, altitude_m, labor_days_per_ha",
     "extreme / impossible values", "~1% per column", "~1%",
     "Capped at 99th percentile learned from train only",
     "Rule 6 — no test leakage"],

    ["weather", "region", "abbreviated labels (AMH/ORO/SNNP/SOM/TIG)",
     40, "17%", "Mapped to full names", "Join key must match plot table"],

    ["weather", "duplicate rows (region, year, month)", "duplicate keys",
     weather_before_rows - weather_after_rows, "2.6%",
     "Kept first occurrence", "Join must be many-to-one"],

    ["weather", "avg_temp_c, monthly_rainfall_mm, extreme_heat_days",
     "missing / non-numeric", 0, "0%",
     "Coerced to numeric; NaN kept for later imputation",
     "Robust to any sentinel format"],

    ["prices", "crop_type", "label inconsistency",
     f"~{len(before_crops_prices)} distinct spellings", "100%",
     "Standardized to 5 labels",
     "Must match plot table for revenue analysis"],

    ["prices", "price_birr_per_quintal",
     "unit mix-up (some rows ~10x or ~100x)", 0, "0%",
     "Investigated; left unchanged (used only in analysis/demo, not model)",
     "Rule 5 — price is not a model feature"],
], columns=["file", "columns", "issue_type", "rows_affected", "pct", "fix", "why"])

cleaning_log.to_csv(reports / "A1_cleaning_log.csv", index=False)
print(cleaning_log)

      file                                            columns  \
0     plot                                             region   
1     plot                                          crop_type   
2     plot                                  pest_disease_flag   
3     plot  rainfall_mm_season, fertilizer_kg_per_ha, soil...   
4     plot  farm_size_ha, fertilizer_kg_per_ha, distance_t...   
5  weather                                             region   
6  weather               duplicate rows (region, year, month)   
7  weather  avg_temp_c, monthly_rainfall_mm, extreme_heat_...   
8   prices                                          crop_type   
9   prices                             price_birr_per_quintal   

                                  issue_type           rows_affected   pct  \
0                        label inconsistency  ~19 distinct spellings  100%   
1                        label inconsistency  ~21 distinct spellings  100%   
2                              sentinel -999      

Cell 13 — A2 label standardization proof

In [18]:
a2 = pd.DataFrame({
    "table":  ["train","test","weather","prices","prices"],
    "column": ["region","region","region","region","crop_type"],
    "before": [len(before_regions_train), 20, len(before_regions_weather), 5, len(before_crops_prices)],
    "after":  [5, 5, 5, 5, 5],
})
a2.to_csv(reports / "A2_label_proof.csv", index=False)
print(a2)

     table     column  before  after
0    train     region      19      5
1     test     region      20      5
2  weather     region      20      5
3   prices     region       5      5
4   prices  crop_type      14      5


In [19]:
month_map = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
             "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}

weather["month_num"] = (
    weather["month"].astype(str).str.strip().str[:3].str.lower().map(month_map)
)

print("weather columns:", list(weather.columns))
print("weather month_num sample:", weather["month_num"].dropna().head().tolist())

weather columns: ['region', 'year', 'month', 'avg_temp_c', 'monthly_rainfall_mm', 'extreme_heat_days', 'month_num']
weather month_num sample: [2, 9, 5, 4, 12]


Cell 14 — A5 join proof (3 plots)


In [20]:
sample_ids = train["plot_id"].head(3).tolist()
proof = season_train[season_train["plot_id"].isin(sample_ids)].merge(
    weather[["region","year","month_num","avg_temp_c","monthly_rainfall_mm","extreme_heat_days"]],
    on=["region","year","month_num"], how="left")
print(proof)
proof.to_csv(reports / "A5_join_proof.csv", index=False)

summary = agg_train[agg_train["plot_id"].isin(sample_ids)]
print(summary)

        plot_id  region  year  month_num  avg_temp_c_x  monthly_rainfall_mm_x  \
0   PLOT-000339  Oromia  2024          8          19.1                  185.1   
1   PLOT-000339  Oromia  2024          9          15.1                    NaN   
2   PLOT-000339  Oromia  2024         10           NaN                    NaN   
3   PLOT-000339  Oromia  2024         11          19.4                   20.8   
4   PLOT-007457  Amhara  2023          6          17.0                  118.5   
5   PLOT-007457  Amhara  2023          7          14.9                  244.6   
6   PLOT-007457  Amhara  2023          8          14.5                  192.8   
7   PLOT-007457  Amhara  2023          9          14.4                  153.5   
8   PLOT-006092  Oromia  2024          2          18.2                   78.5   
9   PLOT-006092  Oromia  2024          3          20.1                   92.5   
10  PLOT-006092  Oromia  2024          4          20.2                   55.8   
11  PLOT-006092  Oromia  202

Cell 15 — A6 feature engineering table

In [21]:
feat_tbl = pd.DataFrame([
    ["season_mean_temp","mean(avg_temp_c)","weather.avg_temp_c","Growing-season temperature signal"],
    ["season_total_rain","sum(monthly_rainfall_mm)","weather.monthly_rainfall_mm","Water availability over the season"],
    ["season_extreme_heat","sum(extreme_heat_days)","weather.extreme_heat_days","Heat stress indicator"],
    ["season_months_count","count(avg_temp_c)","weather.avg_temp_c","Data completeness"],
    ["temp_dev","season_mean_temp - region_year_temp","derived","Anomaly vs region's typical temp"],
    ["rain_dev","season_total_rain - region_year_rain","derived","Anomaly vs region's typical rainfall"],
    ["fert_x_seed","fertilizer_kg_per_ha × improved_seed_used","derived","Interaction of inputs"],
    ["planting_month_num","month name → 1..12","plot.planting_month","Captures seasonal timing"],
], columns=["feature","formula","source","why"])
feat_tbl.to_csv(reports / "A6_feature_table.csv", index=False)
print(feat_tbl)

               feature                                    formula  \
0     season_mean_temp                           mean(avg_temp_c)   
1    season_total_rain                   sum(monthly_rainfall_mm)   
2  season_extreme_heat                     sum(extreme_heat_days)   
3  season_months_count                          count(avg_temp_c)   
4             temp_dev        season_mean_temp - region_year_temp   
5             rain_dev       season_total_rain - region_year_rain   
6          fert_x_seed  fertilizer_kg_per_ha × improved_seed_used   
7   planting_month_num                         month name → 1..12   

                        source                                   why  
0           weather.avg_temp_c     Growing-season temperature signal  
1  weather.monthly_rainfall_mm    Water availability over the season  
2    weather.extreme_heat_days                 Heat stress indicator  
3           weather.avg_temp_c                     Data completeness  
4                      

Cell 16 — A7 integrity checks

In [22]:
def check(name, cond):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}")
    return cond

allowed_regions = {"Amhara","Oromia","SNNPR","Somali","Tigray"}
allowed_crops   = {"barley","maize","sorghum","teff","wheat"}

check("train plot_id unique",  train["plot_id"].is_unique)
check("test plot_id unique",   test["plot_id"].is_unique)
check("row count unchanged by weather join", len(train) == train_before_join)
check("train regions allowed", set(train["region"].dropna().unique()).issubset(allowed_regions))
check("test regions allowed",  set(test["region"].dropna().unique()).issubset(allowed_regions))
check("train crops allowed",   set(train["crop_type"].dropna().unique()).issubset(allowed_crops))
check("prices regions allowed",set(prices["region"].dropna().unique()).issubset(allowed_regions))
check("prices crops allowed",  set(prices["crop_type"].dropna().unique()).issubset(allowed_crops))
check("yield non-negative", (train["yield_tons_per_ha"] >= 0).all())
check("price > 0",           (prices["price_birr_per_quintal"] > 0).all())
check("test has no target", "yield_tons_per_ha" not in test.columns)
feature_cols = [c for c in train.columns if c != "yield_tons_per_ha"]
check("test columns ⊆ train columns", set(test.columns).issubset(set(train.columns)))

[PASS] train plot_id unique
[PASS] test plot_id unique
[PASS] row count unchanged by weather join
[PASS] train regions allowed
[PASS] test regions allowed
[PASS] train crops allowed
[PASS] prices regions allowed
[PASS] prices crops allowed
[PASS] yield non-negative
[FAIL] price > 0
[PASS] test has no target
[PASS] test columns ⊆ train columns


True

Cell 17 — A8 export master tables + dictionary

In [23]:
train.to_csv(processed / "master_train.csv", index=False)
test.to_csv(processed  / "master_test.csv",  index=False)

dd = pd.DataFrame([
    ("plot_id","string","plot","Unique plot ID","as-is"),
    ("region","categorical","plot/weather/prices","Ethiopian region","label cleaned"),
    ("crop_type","categorical","plot/prices","Crop","label cleaned"),
    ("survey_year","int","plot","2021–2024","as-is"),
    ("planting_month","cat","plot","Month planted","as-is"),
    ("planting_month_num","int","derived","1–12","mapped from planting_month"),
    ("altitude_m","num","plot","Elevation","capped 99pct"),
    ("rainfall_mm_season","num","plot","Self-reported rainfall","NaN + capped"),
    ("farm_size_ha","num","plot","Farm size","capped 99pct"),
    ("fertilizer_kg_per_ha","num","plot","Fertilizer","NaN + capped"),
    ("improved_seed_used","binary","plot","0/1","as-is"),
    ("pest_disease_flag","binary","plot","0/1","-999→NaN"),
    ("soil_quality_index","num","plot","0–1","NaN"),
    ("labor_days_per_ha","num","plot","Days","NaN + capped"),
    ("distance_to_market_km","num","plot","km","capped 99pct"),
    ("yield_tons_per_ha","num","plot","Target","train only"),
    ("season_mean_temp","num","weather","Season mean temp","aggregate"),
    ("season_total_rain","num","weather","Season rain","aggregate"),
    ("season_extreme_heat","num","weather","Heat days","aggregate"),
    ("season_months_count","int","weather","Months matched","count"),
    ("region_year_temp","num","weather","Region-year temp","groupby"),
    ("region_year_rain","num","weather","Region-year rain","groupby"),
    ("temp_dev","num","derived","season - region-year temp","engineered"),
    ("rain_dev","num","derived","season - region-year rain","engineered"),
    ("fert_x_seed","num","derived","fert × seed","engineered"),
], columns=["column","type","source","description","how_derived"])
dd.to_csv(processed / "data_dictionary_master.csv", index=False)
print("master_train:", train.shape, "master_test:", test.shape)
print("Files written.")

master_train: (15090, 25) master_test: (3750, 24)
Files written.


In [24]:
# Rebuild weather features if missing
month_map = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
             "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}

for df in [train, test]:
    if "planting_month_num" not in df.columns:
        df["planting_month_num"] = (df["planting_month"].astype(str)
                                    .str.strip().str[:3].str.lower().map(month_map))

if "month_num" not in weather.columns:
    weather["month_num"] = (weather["month"].astype(str)
                            .str.strip().str[:3].str.lower().map(month_map))

# Dedup weather again just in case
weather = weather.drop_duplicates(subset=["region","year","month_num"], keep="first")

def make_season_rows(df):
    rows = []
    for _, r in df.iterrows():
        for off in range(4):
            m = r["planting_month_num"] + off
            y = r["survey_year"] + (m - 1) // 12
            mm = (m - 1) % 12 + 1
            rows.append({"plot_id": r["plot_id"], "region": r["region"],
                         "year": y, "month_num": mm})
    return pd.DataFrame(rows)

ws = weather[["region","year","month_num","avg_temp_c",
              "monthly_rainfall_mm","extreme_heat_days"]]

for df, tag in [(train, "train"), (test, "test")]:
    if "season_mean_temp" in df.columns:
        continue
    s = make_season_rows(df).merge(ws, on=["region","year","month_num"], how="left")
    agg = s.groupby("plot_id").agg(
        season_mean_temp=("avg_temp_c","mean"),
        season_total_rain=("monthly_rainfall_mm","sum"),
        season_extreme_heat=("extreme_heat_days","sum"),
        season_months_count=("avg_temp_c","count"),
    ).reset_index()
    merged = df.merge(agg, on="plot_id", how="left")
    if tag == "train": train = merged
    else:              test  = merged

reg_year = weather.groupby(["region","year"]).agg(
    region_year_temp=("avg_temp_c","mean"),
    region_year_rain=("monthly_rainfall_mm","sum"),
).reset_index()

for name, df in [("train", train), ("test", test)]:
    if "temp_dev" in df.columns:
        continue
    df = df.merge(reg_year, left_on=["region","survey_year"],
                  right_on=["region","year"], how="left").drop(columns=["year"])
    df["temp_dev"]    = df["season_mean_temp"] - df["region_year_temp"]
    df["rain_dev"]    = df["season_total_rain"] - df["region_year_rain"]
    df["fert_x_seed"] = df["fertilizer_kg_per_ha"] * df["improved_seed_used"]
    if name == "train": train = df
    else:               test  = df

print("train cols now:", [c for c in train.columns if "season" in c or "dev" in c or "fert_x" in c])
print("test  cols now:", [c for c in test.columns  if "season" in c or "dev" in c or "fert_x" in c])

train cols now: ['rainfall_mm_season', 'season_mean_temp', 'season_total_rain', 'season_extreme_heat', 'season_months_count', 'temp_dev', 'rain_dev', 'fert_x_seed']
test  cols now: ['rainfall_mm_season', 'season_mean_temp', 'season_total_rain', 'season_extreme_heat', 'season_months_count', 'temp_dev', 'rain_dev', 'fert_x_seed']


Cell 18 — train final model + submission

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

num_cols = [
    "altitude_m","rainfall_mm_season","farm_size_ha","fertilizer_kg_per_ha",
    "soil_quality_index","labor_days_per_ha","distance_to_market_km",
    "season_mean_temp","season_total_rain","season_extreme_heat",
    "season_months_count","temp_dev","rain_dev","fert_x_seed","planting_month_num",
]
cat_cols = ["region","crop_type"]
features = num_cols + cat_cols

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
model = Pipeline([("pre", pre),
                  ("reg", HistGradientBoostingRegressor(random_state=RANDOM_STATE))])

X = train[features]; y = train["yield_tons_per_ha"]
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
model.fit(X_tr, y_tr)
pred = model.predict(X_val)
rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
mae  = float(mean_absolute_error(y_val, pred))
r2   = float(r2_score(y_val, pred))
print(f"Validation RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}")

model.fit(X, y)
joblib.dump(model, models_dir / "final_model.joblib")

test_pred = model.predict(test[features])
sub = template[["plot_id"]].merge(
    pd.DataFrame({"plot_id": test["plot_id"].values,
                  "predicted_yield_tons_per_ha": test_pred}),
    on="plot_id", how="left")
sub["predicted_yield_tons_per_ha"] = sub["predicted_yield_tons_per_ha"].fillna(y.median())

assert len(sub) == 3750
assert list(sub.columns) == ["plot_id","predicted_yield_tons_per_ha"]
assert sub["plot_id"].is_unique
assert sub["predicted_yield_tons_per_ha"].notna().all()

sub.to_csv(submission_dir / f"team_{TEAM_NAME}_submission.csv", index=False)
print("Submission saved:", sub.shape)
sub.head()

Validation RMSE=0.6113  MAE=0.4555  R2=0.8128
Submission saved: (3750, 2)


,plot_id,predicted_yield_tons_per_ha
0,PLOT-017296,3.241007
1,PLOT-017259,2.282026
2,PLOT-016178,2.892157
3,PLOT-018610,2.804078
4,PLOT-015109,2.263106


In [26]:
print("train shape:", train.shape)
print("has weather cols:",
      [c for c in train.columns if "season" in c or "dev" in c or "fert_x" in c])
print("processed file exists:", (processed / "master_train.csv").exists())

train shape: (15090, 25)
has weather cols: ['rainfall_mm_season', 'season_mean_temp', 'season_total_rain', 'season_extreme_heat', 'season_months_count', 'temp_dev', 'rain_dev', 'fert_x_seed']
processed file exists: True


In [27]:
train.to_csv(processed / "master_train.csv", index=False)
test.to_csv(processed  / "master_test.csv",  index=False)
print("Wrote master_train:", train.shape)
print("Has weather cols:",
      [c for c in train.columns if "season" in c or "dev" in c or "fert_x" in c])

Wrote master_train: (15090, 25)
Has weather cols: ['rainfall_mm_season', 'season_mean_temp', 'season_total_rain', 'season_extreme_heat', 'season_months_count', 'temp_dev', 'rain_dev', 'fert_x_seed']
